In [2]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
import swifter
from functools import reduce
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.DataStructs import TanimotoSimilarity
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect
from rdkit.Chem import Descriptors

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {"toImageButtonOptions": {"format": "png", "scale": 5}}

sys.path.append(os.path.abspath(".."))
from new_geass import Geass

sys.path.append(os.path.abspath(R"C:\\Users\\Idener\\Sync\\CODE"))
from AIDDTools import fingerprint_tools as fpt
clear_output()

In [41]:
# Create the unknown unit library
df_NPA = pd.read_pickle('NPA_analysis/Database_NPA_fragmented.pkl')
df_BUL = pd.read_pickle('data/Database_BUL.pkl')

df_UUL = df_NPA[~df_NPA['Frag_IDs'].str.startswith(('C', 'L'))].reset_index(drop=True)
df_UUL = df_UUL.drop_duplicates(subset=['ISOSMILES']).reset_index(drop=True)
df_UUL['ISOSMILES'] = df_UUL['Fragments']
df_UUL = df_UUL.drop(columns=['ID', 'Fragments']).rename(columns={'Frag_IDs': 'ID_Similar'})
df_UUL['ID_Similar'] = df_UUL['ID_Similar'].apply(lambda x: x.replace('~', '').replace('[', '').replace(']', ''))
df_UUL = pd.merge(df_UUL, df_BUL[['ID', 'ISOSMILES']], left_on='ID_Similar', right_on='ID', how='left')
df_UUL = df_UUL.rename(columns={'ISOSMILES_x': 'ISOSMILES', 'ISOSMILES_y': 'ISOSMILES_Similar'})

df_UUL['InChI'] = df_UUL['ISOSMILES'].apply(lambda x: Chem.MolToInchi(Chem.MolFromSmiles(x)))
df_UUL = df_UUL[['ISOSMILES', 'InChI', 'ID_Similar', 'ISOSMILES_Similar', 'Probability']]

df_UUL.to_csv('data/Database_UUL.csv', index=False)
df_UUL.to_pickle('data/Database_UUL.pkl')
df_UUL

,ISOSMILES,InChI,ID_Similar,ISOSMILES_Similar,Probability
0,CC[C@H](C)[C@H](N)CO,"InChI=1S/C6H15NO/c1-3-5(2)6(7)4-8/h5-6,8H,3-4,...",LMFA07010874,CCCC(=O)OCC,0.56
1,C=C(C)[C@@H]1C[C@@]2(C)C(=CC1=O)CC[C@@H](O)[C@...,InChI=1S/C15H22O2/c1-9(2)12-8-15(4)10(3)13(16)...,C09627,C=C(C)[C@@H]1CC=C2[C@H](O)C[C@@H](O)[C@@H](C)[...,0.92
2,O=C(O)C1=CC(O)C(O)C(O)O1,InChI=1S/C6H8O6/c7-2-1-3(5(9)10)12-6(11)4(2)8/...,C22712,O=C(O)C1=C[C@H](O)[C@@H](O)C(O)O1,0.98
3,C[C@H]1O[C@@H](O)CCC1O,"InChI=1S/C6H12O3/c1-4-5(7)2-3-6(8)9-4/h4-8H,2-...",C03348,C[C@@H]1O[C@@H](O)[C@@H](O)C[C@@H]1O,0.97
4,C[C@]12C[C@H](O)C[C@H](O1)c1c(cc3oc4cccc(O)c4c...,InChI=1S/C19H16O7/c1-19-7-8(20)5-12(25-19)15-1...,LMPK13040026,C[C@H](O)Cc1cc(O)c2c(c1)C(=O)c1cccc(O)c1C2=O,0.75
...,...,...,...,...,...
20572,C[C@@H](O)C[C@@H](O)[C@H](Cl)/C=C/C(=O)O,InChI=1S/C8H13ClO4/c1-5(10)4-7(11)6(9)2-3-8(12...,LMFA01020402,CC[C@@H](O)[C@@H](C)/C=C/C(=O)O,0.66
20573,Cc1ccc([C@]2(C)CCCC2(C)C)c(O)c1O,InChI=1S/C15H22O2/c1-10-6-7-11(13(17)12(10)16)...,LMPR0103160009,Cc1ccc(O)c([C@@]2(C)CCCC2(C)C)c1,0.99
20574,Cc1cc([C@@]2(C)CCCC2(C)C)c(O)cc1O,InChI=1S/C15H22O2/c1-10-8-11(13(17)9-12(10)16)...,LMPR0103160011,Cc1cc(O)c(O)c([C@@]2(C)CCCC2(C)C)c1,0.98
20575,C[C@@H](O)[C@H]1[C@@](C)(O)CC(=O)[C@]23O[C@]12...,"InChI=1S/C34H34O14/c1-11(35)23-29(3,45)9-17(37...",LMPK12112244,C[C@@H]1O[C@@H](O[C@H]2[C@H](Oc3c(-c4ccc(O)c(O...,0.28


## Analysis of the unknown unit library

In [42]:
# Create histogram with 0.10 bin width
fig = px.histogram(
    df_UUL,
    x='Probability',
    nbins=int(1 / 0.05),  # 20 bins of width 0.05
    labels={'Probability': 'Predicted similarity'},
    color_discrete_sequence=['#009b73'])

# Style
fig.update_layout(
    bargap=0.05,
    width=800,
    height=450,
    xaxis=dict(tickmode='array', tickvals=[i/10 for i in range(11)]),
    yaxis_title='Number of compounds',
    font=dict(family='sans-serif'))

fig.show()

print(f"""======= UUL Summary =======

UUs with similarity annotation (sim ≥ 0.85): {len(df_UUL[df_UUL['Probability'] >= 0.85])}
UUs with no annotation (sim < 0.85):         {len(df_UUL[df_UUL['Probability'] < 0.85])}
Total UUs:                                   {len(df_UUL)}
""")

======= UUL Summary =======

UUs with similarity annotation (sim ≥ 0.85): 9036
UUs with no annotation (sim < 0.85):         11541
Total UUs:                                   20577



In [ ]:
# Plot histogram using Plotly
fig = px.bar(
    hist_df,
    x='Bin',
    y='Count',
    text='Text',
    color='Color',
    color_discrete_map='identity',  # use exact color names
    labels={'Bin': f'Similarity Range ({method})', 'Count': 'Frequency'})

fig.update_layout(
    xaxis_title= f'Similarity Range ({method})',
    yaxis_title='Count',
    font=dict(family="sans-serif"),
    bargap=0.1,
    width=800,
    height=400)

fig.show(config={"toImageButtonOptions": {"format":"svg", "scale":5}})



In [ ]:
df_UUL_grouped = df_UUL.groupby('ISOSMILES_Similar').agg(list).reset_index()
df_UUL_grouped['Num_compounds'] = df_UUL_grouped['ID'].apply(len)
df_UUL_grouped['ID_Similar'] = df_UUL_grouped['ID_Similar'].apply(lambda x: list(set(x))[0])

bin_edges = range(0, df_UUL_grouped['Num_compounds'].max() + 10, 10)
df_UUL_grouped['Compound_bin'] = pd.cut(df_UUL_grouped['Num_compounds'], bins=bin_edges, right=False)

# Count the number of ISOSMILES_Similar entries in each bin
bin_counts = df_UUL_grouped['Compound_bin'].value_counts().sort_index()

# Create a bar chart
fig = px.bar(
    x=bin_counts.index.astype(str), 
    y=bin_counts.values,
    labels={'x': 'Number of Compounds (Binned)', 'y': 'Count of ISOSMILES_Similar Groups'},
)

fig.show()